В чем суть - Polaris дает внутри себя отдельные гранты на таблицы и схемы. Если Trino берет один общий грант на S3 и от Polaris узнает только локацию последней метадаты, то PyIceberg ведет себя более честно, и спрашивает у Polaris, можно ли ему прочитать/записать в опредлеенную таблицу.
Отдельная проблема в том, что нет ни команды GRANT, ни какого-то удобного фронтенда, поэтому приходится делать обвязки вокруг curl или requests

In [1]:
import requests
from typing import List

In [2]:
POLARIS_CATALOG_URI = "http://localhost:8181/api/catalog"
POLARIS_MANAGEMENT_URI = "http://localhost:8181/api/management"

REALM = "default-realm"

# Это имя catalog / warehouse в Polaris.
# Если в PyIceberg у тебя warehouse="default", оставь default.
# Если warehouse="rest", поставь rest.
POLARIS_CATALOG = "default"

# Catalog role, которому выдаём права
CATALOG_ROLE = "catalog_admin"

CLIENT_ID = "root"
CLIENT_SECRET = "polaris-root-secret-change-me"

SCOPE = "PRINCIPAL_ROLE:ALL"

In [3]:
POLARIS_CATALOG_URI = "http://polaris:8181/api/catalog"
POLARIS_MANAGEMENT_URI = "http://polaris:8181/api/management"

In [4]:
def get_polaris_token(
    polaris_catalog_uri: str = POLARIS_CATALOG_URI,
    realm: str = REALM,
    client_id: str = CLIENT_ID,
    client_secret: str = CLIENT_SECRET,
    scope: str = SCOPE,
) -> str:
    response = requests.post(
        f"{polaris_catalog_uri}/v1/oauth/tokens",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Polaris-Realm": realm,
        },
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": scope,
        },
        timeout=30,
    )

    if not response.ok:
        print("Failed to get token")
        print("Status:", response.status_code)
        print("Body:", response.text)
        response.raise_for_status()

    token = response.json()["access_token"]
    return token

In [5]:
token = get_polaris_token()

print(token)

principal:root;password:polaris-root-secret-change-me;realm:default-realm;role:ALL


In [6]:
def grant_table_privilege(
    schema: str,
    table: str,
    privilege: str,
    token: str,
    polaris_management_uri: str = POLARIS_MANAGEMENT_URI,
    realm: str = REALM,
    polaris_catalog: str = POLARIS_CATALOG,
    catalog_role: str = CATALOG_ROLE,
) -> None:
    url = (
        f"{polaris_management_uri}/v1/catalogs/"
        f"{polaris_catalog}/catalog-roles/{catalog_role}/grants"
    )

    headers = {
        "Authorization": f"Bearer {token}",
        "Polaris-Realm": realm,
        "Content-Type": "application/json",
    }

    payload = {
        "grant": {
            "type": "table",
            "namespace": [schema],
            "table": table,
            "privilege": privilege,
        }
    }

    response = requests.put(
        url,
        headers=headers,
        json=payload,
        timeout=30,
    )

    # Fallback для версий Polaris, где поле называется tableName
    if response.status_code == 400:
        payload = {
            "grant": {
                "type": "table",
                "namespace": [schema],
                "tableName": table,
                "privilege": privilege,
            }
        }

        response = requests.put(
            url,
            headers=headers,
            json=payload,
            timeout=30,
        )

    if not response.ok:
        print(f"Failed to grant {privilege} on {schema}.{table}")
        print("URL:", url)
        print("Payload:", payload)
        print("Status:", response.status_code)
        print("Body:", response.text)
        response.raise_for_status()

    print(f"Granted {privilege} on {schema}.{table} to catalog role {catalog_role}")

In [8]:
def grant_table_privileges(
    schema: str,
    table: str,
    privileges: List[str] | None = None,
    token: str | None = None,
) -> None:
    if privileges is None:
        privileges = [
            "TABLE_READ_DATA",
            "TABLE_WRITE_DATA",
        ]

    if token is None:
        token = get_polaris_token()

    for privilege in privileges:
        grant_table_privilege(
            schema=schema,
            table=table,
            privilege=privilege,
            token=token,
        )

In [9]:
grant_table_privileges(
    schema="my",
    table="events",
)

Granted TABLE_READ_DATA on my.events to catalog role catalog_admin
Granted TABLE_WRITE_DATA on my.events to catalog role catalog_admin


In [ ]:
# То же на схемы (каталоги в терминологии Поларис)

In [10]:
def grant_namespace_privilege(
    schema: str,
    privilege: str,
    token: str,
    polaris_management_uri: str = POLARIS_MANAGEMENT_URI,
    realm: str = REALM,
    polaris_catalog: str = POLARIS_CATALOG,
    catalog_role: str = CATALOG_ROLE,
) -> None:
    url = (
        f"{polaris_management_uri}/v1/catalogs/"
        f"{polaris_catalog}/catalog-roles/{catalog_role}/grants"
    )

    headers = {
        "Authorization": f"Bearer {token}",
        "Polaris-Realm": realm,
        "Content-Type": "application/json",
    }

    payload = {
        "grant": {
            "type": "namespace",
            "namespace": [schema],
            "privilege": privilege,
        }
    }

    response = requests.put(
        url,
        headers=headers,
        json=payload,
        timeout=30,
    )

    if not response.ok:
        print(f"Failed to grant {privilege} on namespace {schema}")
        print("URL:", url)
        print("Payload:", payload)
        print("Status:", response.status_code)
        print("Body:", response.text)
        response.raise_for_status()

    print(
        f"Granted {privilege} on namespace {schema} "
        f"to catalog role {catalog_role}"
    )

In [12]:
token = get_polaris_token()

for privilege in [
    "TABLE_CREATE",
    "NAMESPACE_LIST",
    "NAMESPACE_READ_PROPERTIES",
    "NAMESPACE_WRITE_PROPERTIES",
]:
    grant_namespace_privilege(
        schema="my",
        privilege=privilege,
        token=token,
    )

Granted TABLE_CREATE on namespace my to catalog role catalog_admin
Granted NAMESPACE_LIST on namespace my to catalog role catalog_admin
Granted NAMESPACE_READ_PROPERTIES on namespace my to catalog role catalog_admin
Granted NAMESPACE_WRITE_PROPERTIES on namespace my to catalog role catalog_admin


In [ ]:
#  Доступные права в поларис

[
  CATALOG_MANAGE_ACCESS,
  CATALOG_MANAGE_CONTENT,
  CATALOG_MANAGE_METADATA,
  NAMESPACE_CREATE,
  TABLE_CREATE,
  VIEW_CREATE,
  NAMESPACE_DROP,
  TABLE_DROP,
  VIEW_DROP,
  NAMESPACE_LIST,
  TABLE_LIST,
  VIEW_LIST,
  NAMESPACE_READ_PROPERTIES,
  TABLE_READ_PROPERTIES,
  VIEW_READ_PROPERTIES,
  NAMESPACE_WRITE_PROPERTIES,
  TABLE_WRITE_PROPERTIES,
  VIEW_WRITE_PROPERTIES,
  TABLE_READ_DATA,
  TABLE_WRITE_DATA,
  NAMESPACE_FULL_METADATA,
  TABLE_FULL_METADATA,
  VIEW_FULL_METADATA
]

In [14]:
def grant_catalog_privilege(
    privilege: str,
    token: str,
    polaris_management_uri: str = POLARIS_MANAGEMENT_URI,
    realm: str = REALM,
    polaris_catalog: str = POLARIS_CATALOG,
    catalog_role: str = CATALOG_ROLE,
) -> None:
    url = (
        f"{polaris_management_uri}/v1/catalogs/"
        f"{polaris_catalog}/catalog-roles/{catalog_role}/grants"
    )

    headers = {
        "Authorization": f"Bearer {token}",
        "Polaris-Realm": realm,
        "Content-Type": "application/json",
    }

    payload = {
        "grant": {
            "type": "catalog",
            "privilege": privilege,
        }
    }

    response = requests.put(
        url,
        headers=headers,
        json=payload,
        timeout=30,
    )

    if not response.ok:
        print(f"Failed to grant {privilege} on catalog {polaris_catalog}")
        print("URL:", url)
        print("Payload:", payload)
        print("Status:", response.status_code)
        print("Body:", response.text)
        response.raise_for_status()

    print(
        f"Granted {privilege} on catalog {polaris_catalog} "
        f"to catalog role {catalog_role}"
    )

In [15]:
token = get_polaris_token()

for privilege in [
    "TABLE_CREATE",
    "TABLE_LIST",
    "TABLE_READ_PROPERTIES",
    "TABLE_WRITE_PROPERTIES",
    "TABLE_READ_DATA",
    "TABLE_WRITE_DATA",
    "TABLE_FULL_METADATA",
    "NAMESPACE_LIST",
    "NAMESPACE_READ_PROPERTIES",
    "NAMESPACE_WRITE_PROPERTIES",
]:
    grant_namespace_privilege(
        schema="my",
        privilege=privilege,
        token=token,
    )

for privilege in [
    "CATALOG_MANAGE_CONTENT",
    "CATALOG_MANAGE_METADATA",
]:
    grant_catalog_privilege(
        privilege=privilege,
        token=token,
    )

Granted TABLE_CREATE on namespace my to catalog role catalog_admin
Granted TABLE_LIST on namespace my to catalog role catalog_admin
Granted TABLE_READ_PROPERTIES on namespace my to catalog role catalog_admin
Granted TABLE_WRITE_PROPERTIES on namespace my to catalog role catalog_admin
Granted TABLE_READ_DATA on namespace my to catalog role catalog_admin
Granted TABLE_WRITE_DATA on namespace my to catalog role catalog_admin
Granted TABLE_FULL_METADATA on namespace my to catalog role catalog_admin
Granted NAMESPACE_LIST on namespace my to catalog role catalog_admin
Granted NAMESPACE_READ_PROPERTIES on namespace my to catalog role catalog_admin
Granted NAMESPACE_WRITE_PROPERTIES on namespace my to catalog role catalog_admin
Granted CATALOG_MANAGE_CONTENT on catalog default to catalog role catalog_admin
Granted CATALOG_MANAGE_METADATA on catalog default to catalog role catalog_admin
